# Events and Model State

CSC-239 · Module 12 · Lesson 3 of 4

You can build a readable native JavaFX layout. Now you will connect a user action to an ordinary Java object and display the resulting state. You will test with both pointer and keyboard input.

Use a fresh **Java** kernel and run the supplied setup. Open the Workspace **Desktop** view to use the native controls. Run each complete example from its beginning before a new test case so the model and controls return to their starting state.


## Learning Goals

- Register a handler that changes an ordinary Java model and updates a visible Label.
- Use pointer and keyboard input to reach a boundary state and verify the button’s disabled behavior.


## Why This Matters

A reservation desk has three seats available. A Reserve one button should reduce that count, show the new value, and stop accepting reservations when no seats remain.


## Check Your Starting Point

Explain how a private field and a public method protect an object’s state. Recall that a lambda supplies behavior without immediately running its body. Explain why an effectively final local reference can still refer to an object whose state changes. Review the difference between a VBox’s children and the model objects used by the program.

**My explanation:**


## Concept

### Let a control request an action

A **Button control** requests an action when a user activates it. `new Button("Reserve one")` creates a button whose visible text describes that action. Creating it does not reserve a seat.

An **action event** is a notification that a control’s requested action occurred. A pointer click can cause an action event. A focused button can also be activated from the keyboard. Connect behavior to the action event so both input methods can use the same operation.

**Event-handler registration** connects behavior to a future event. `reserve.setOnAction(event -> { ... })` registers a handler: the lambda body will run when the button’s action occurs. A handler is also called a callback: behavior JavaFX calls when a matching event occurs. The `event` parameter receives the notification object. We do not need to read that object’s fields for this example.

Watch the difference between registering behavior and calling a model method immediately. A call such as `model.reserve()` during construction would change the count before the user does anything. Inside the registered handler, it happens in response to activation.

### Separate the stored state from its display

**Model and view separation** keeps application state in an ordinary Java object and displays it through controls. The model knows the remaining count. The Label presents a readable description of that count. The model does not need to know where the Label is placed.

For this lesson, `SeatCounter` has a private remaining field, a getter, and a guarded reserve method. Its constructor receives a nonnegative starting count; that is a caller requirement for this small class. The guard prevents reserve from decreasing a zero count. It does not validate a negative constructor argument.

The Label’s text is not the authoritative count. Do not parse the label’s sentence to decide how many seats remain. Read the model, apply its operation, and then update the label from the new model value.

### Change state and then refresh the view

An **event-driven state update** changes model state in response to an event, then refreshes the visible result. The reserve handler performs three related steps:

1. Ask the model to reserve one seat.
2. Set the Label text from the model’s updated count.
3. Decide whether the button should accept another action.

If you update only the model, the label can keep showing an old value. If you update only the label, the model can still hold the old count. Test the sequence of actions and visible results, then test the ordinary model separately.

JavaFX calls action handlers on its application thread. That gives these short handlers the correct place to update live controls. Do not add file reads, long searches, or waiting loops to a handler. The same thread must remain free to respond to the interface.

### Enable actions only when they make sense

**Control enablement** allows or prevents a control’s user action according to current state. `reserve.setDisable(true)` disables the Reserve one button; `setDisable(false)` allows it again.

The worked handler disables the button when remaining is zero. The visible label also says Remaining: 0, so the boundary is communicated in words. The disabled appearance is an additional cue, not the only explanation.

Keep the model’s guard too. Disabling a button improves the interface, while the guard protects the object if another caller invokes reserve directly. Those two checks serve different parts of the program.

### Use focus and activate from the keyboard

**Keyboard focus and activation** means choosing which control receives keyboard input and then requesting its action. Focus is the currently selected destination for keyboard input; it is not the same as being enabled.

On the Workspace Desktop, use **Tab** to move forward among reachable controls and **Shift+Tab** to move backward. A visible focus outline helps identify the selected button. With that button focused, press **Space** to activate it.

In the independent Point Counter, reach the limit with Add two, then activate Reset. After Reset has focus, Shift+Tab selects Add two and Space adds two points. Observe the result. A disabled Add two control should not accept another action at the limit, and Reset should allow it again.

### Read the complete mechanism

After the supplied setup, this small program uses one object for state and two controls for interaction:

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class ClickCount {
    private int count;
    public ClickCount() { count = 0; }
    public void addOne() { count++; }
    public int getCount() { return count; }
}
Fx.closeWindows();
Fx.run(() -> {
    ClickCount model = new ClickCount();
    Label status = new Label("Clicks: " + model.getCount());
    Button add = new Button("Add one");
    add.setOnAction(event -> {
        model.addOne();
        status.setText("Clicks: " + model.getCount());
    });
    VBox root = new VBox(12, status, add);
    root.setPadding(new Insets(20));
    Stage stage = new Stage();
    stage.setTitle("Click Counter");
    stage.setScene(new Scene(root, 340, 220));
    stage.show();
    System.out.println("Initial count: " + model.getCount());
});
```

The notebook initially prints `Initial count: 0`, and the Desktop initially shows Clicks: 0. Each activation changes the visible count. The initial printed line does not change afterward. A completed notebook cell means the interface was constructed; it does not mean the later user interactions have been tested.

The lambda captures the model and status references. Neither local reference is reassigned, so it can be used by the lambda. Calling addOne or setText changes the referenced object’s state; it does not assign a new value to the local reference.


### Prepare this kernel

This is supplied course support for running JavaFX inside IJava. Run it once after starting or restarting this notebook's Java kernel. The message `FX ready` means the support has initialized JavaFX and completed an operation on its application thread. Open the Workspace **Desktop** view to see the windows created by later cells.

`Fx.run(() -> { ... })` performs the enclosed UI work on the JavaFX Application Thread and waits for that short operation to finish. Use it for reading as well as changing a live window or its controls. `Fx.closeWindows()` hides the windows created by this kernel before another example opens its own. `Fx.start()` is safe to call again; it keeps JavaFX available after the last window closes.

The implementation below is provided runtime support. You do not need to write its thread-coordination machinery for this lesson. A thread is one sequence of execution; Module 13 studies how to coordinate more than one. Here your responsibility is to use the documented support operations and keep UI work short. The support uses a completion signal, a time limit, and an error holder so a later cell does not silently continue after unfinished or failed UI work.

Do not call `Platform.exit()` during notebook practice. That ends the toolkit for this kernel; restart the kernel and rerun setup if you do so. Closing a window is different from ending the toolkit. If setup reports a display error, check that the Workspace Desktop is running, then restart the kernel and rerun setup. A JavaFX window appears in the Desktop, not as an inline notebook control.


In [ ]:
import javafx.application.Platform;
import javafx.stage.Window;
import java.util.ArrayList;
import java.util.concurrent.CountDownLatch;
import java.util.concurrent.TimeUnit;
import java.util.concurrent.atomic.AtomicReference;
class Fx {
    static void run(Runnable action) throws InterruptedException {
        if (Platform.isFxApplicationThread()) {
            action.run();
            return;
        }
        CountDownLatch done = new CountDownLatch(1);
        AtomicReference<Throwable> failure = new AtomicReference<Throwable>();
        Platform.runLater(() -> {
            try { action.run(); }
            catch (Throwable error) { failure.set(error); }
            finally { done.countDown(); }
        });
        if (!done.await(10, TimeUnit.SECONDS)) {
            throw new IllegalStateException("FX operation timed out; restart the kernel.");
        }
        if (failure.get() != null) { throw new RuntimeException(failure.get()); }
    }
    static void start() throws InterruptedException {
        try { Platform.startup(() -> Platform.setImplicitExit(false)); }
        catch (IllegalStateException alreadyStarted) {
            // This call is also safe when this kernel already started JavaFX.
        }
        run(() -> Platform.setImplicitExit(false));
    }
    static void closeWindows() throws InterruptedException {
        run(() -> {
            for (Window window : new ArrayList<Window>(Window.getWindows())) {
                window.hide();
            }
        });
    }
}
Fx.start();
System.out.println("FX ready");


## Video Demonstration

Predict the label after each reservation. Watch the native action change the model, update the label, and disable the button at zero. Compare the initial console report with later visible UI state.

<video controls preload="metadata" width="960">
  <source src="media/03_events_and_model_state/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/03_events_and_model_state/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the events and model state demonstration transcript](media/03_events_and_model_state/transcript.md).


## Worked Example

**Subgoal 1: create model and controls.** Start with three seats, a matching Label, and a Reserve one button.

**Subgoal 2: register the response.** Reserve, refresh the label, and update enablement inside one action handler.

**Subgoal 3: test the boundary.** Activate three times, check Remaining: 0, and try the disabled action again.


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(3);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Seat Counter");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});


Expected output:

```text
Counter ready: 3
```

Construction reports three seats and shows Remaining: 3. The first three activations display 2, 1, and 0. The third also disables Reserve one. A later attempt through that disabled control leaves the count at zero. The model’s own guard also prevents a direct reserve call from making the count negative.


## Predict, Run, Trace, and Explain

### Predict a reservation sequence

Before running, predict the initial console line and visible label. Then predict the label and button state after one pointer click, Space on the focused Reserve one button, and one more click attempt. Identify whether registering the handler uses a seat. Keep your original predictions for comparison.

My initial console and Label predictions:

Does registration use a seat? Why?

After the pointer click:

After Space on the focused button:

After the extra click attempt:


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});


Run the complete prediction program in the Workspace and open its Desktop. Record the initial console line and visible state. Click Reserve one once. With that button focused, press Space once; use Tab first if needed to make its keyboard focus visible. Attempt one more native click on the button. Record the label and enabled or disabled state after each action. Check whether the console adds any lines. Close the window with Alt+F4 and confirm that it disappears. Compare these observations with your original predictions and explain any difference before opening the answer.

My original prediction is retained above.

Actual initial console and Desktop:

After pointer click:

After focused Space:

After the extra click attempt:

New console lines, if any:

Native close observed:

My post-run explanation:

### Separate model, label and handler

Use your recorded actions to trace the model method, label refresh and button state in order. Which object owns the remaining count? Why does changing that value require a separate Label update? Explain why the lambda can change object state while its captured local references stay unchanged. Identify where the initial construction and later callback perform their UI work.

Object that owns the count:

For each successful action: model change → Label update → button state:

Why the Label needs a refresh:

Captured references versus object state:

Where the UI work runs:

<details>
<summary>Show answer</summary>

The initial console prints `Counter ready: 2`. Reservation Desk starts with `Remaining: 2` and an enabled Reserve one button. Registering the lambda with `setOnAction` stores behavior for a future action; it does not call `model.reserve()` during construction. A pointer click changes the model to 1, refreshes the label to `Remaining: 1`, and leaves the button enabled. With Reserve one focused, Space requests the same action: the model reaches 0, the label becomes `Remaining: 0`, and the button becomes disabled. Another native click on that disabled control requests no handler action, so the label stays at zero. These later UI actions add no console output. The model values follow from the exact method and callback statements; the label and button state are the visible observations. The class also checks `remaining > 0` before decrementing. The model guard and the disabled control serve different purposes: one protects the stored value when the method is called, and the other controls whether the user can request the action through this button. The `SeatCounter` object owns the private remaining value. The Label displays text copied from the model; it does not automatically track later field changes. The handler first calls the model method, then refreshes the Label, then sets the button state from the new remaining value. Its lambda captures the model and control references. Those local variables keep referring to the same objects while the objects change state. The supplied `Fx.run` performs construction and initial UI reads on the JavaFX Application Thread; the registered action callback also performs its UI updates there. Both pointer activation and Space on the focused button reach that registered behavior.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 2
```

Common error: Treating handler registration as an immediate method call. Assuming later button actions print another initial report. Predicting button state without tracing the model update.

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete one event callback

Replace the four uppercase placeholders in the displayed draft. Use each method name once: `setText`, `setDisable`, `reserve`, `setOnAction`. Keep the model update before the Label refresh and button check. Write the complete repaired program in the work cell, then test the same pointer, focused Space and disabled-click sequence. Close the window and explain why this order matters.

This sample is for repair:

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.HANDLER_METHOD(event -> {
        model.MODEL_UPDATE();
        status.LABEL_UPDATE("Remaining: " + model.getRemaining());
        reserve.BUTTON_STATE(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```


My four replacements and reasons:

My predicted action sequence:

Actual Desktop observations and native close:

Why model update must precede display refresh:

<details>
<summary>Show answer</summary>

Use `setOnAction` to register the callback, `reserve` to update the model, `setText` to refresh the Label and `setDisable` to set the control state. The initial console prints `Counter ready: 2`. Reservation Desk starts with `Remaining: 2` and an enabled Reserve one button. Registering the lambda with `setOnAction` stores behavior for a future action; it does not call `model.reserve()` during construction. A pointer click changes the model to 1, refreshes the label to `Remaining: 1`, and leaves the button enabled. With Reserve one focused, Space requests the same action: the model reaches 0, the label becomes `Remaining: 0`, and the button becomes disabled. Another native click on that disabled control requests no handler action, so the label stays at zero. These later UI actions add no console output. The model values follow from the exact method and callback statements; the label and button state are the visible observations. The class also checks `remaining > 0` before decrementing. The model guard and the disabled control serve different purposes: one protects the stored value when the method is called, and the other controls whether the user can request the action through this button.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 2
```

Common error: Putting a control method on the model object. Reading the old model value before the reservation. Leaving placeholder names in runnable code.

</details>


### Set the initial button state

The complete starter still begins with two seats. Modify it to begin with zero seats and make Reserve one disabled before any user action. Add an initial `setDisable` call after the Button is created, using the same model-based condition as the handler. Keep the handler check for later updates. Predict, run and inspect the initial button state; attempt a native click and close the window. Then recreate the complete modified program with one starting seat. Focus Reserve one and press Space, attempt an extra click after it disables, and close the window. Explain why checking only inside the handler misses the initial zero case.


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});


My zero-start prediction:

Actual initial disabled state and click attempt:

My one-start prediction:

Actual focused Space and extra-click results:

Why an initial check is needed:

Native close for both programs:

<details>
<summary>Show answer</summary>

Add `reserve.setDisable(model.getRemaining() == 0);` immediately after constructing Reserve one. With zero seats, initial stdout is `Counter ready: 0`; the Label reads `Remaining: 0`, and the button is already disabled. A click attempt changes nothing. The handler has not run yet, so its later check alone cannot set this initial state. The one-seat comparison starts enabled, reaches zero through focused Space, refreshes the Label and disables the button. Both programs close normally. The model and handler definitions are otherwise unchanged.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(0);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setDisable(model.getRemaining() == 0);
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 0
```

Common error: Adding the initial check before constructing the Button. Keeping the button enabled at zero until its first event. Removing the handler check after adding the initial check.

**Additional test: `modify_initial_one`.** A fresh one-seat program prints `Counter ready: 1`, begins with `Remaining: 1` and an enabled button, then focused Space reaches `Remaining: 0` and disables it. The extra native click leaves zero unchanged. This checks both the initial nonzero state and the later boundary transition.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(1);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setDisable(model.getRemaining() == 0);
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 1
```

</details>


### Repair a stale reservation label

The displayed draft keeps the reservation method and button check but omits one UI update. Predict the visible label after a click and then Space on the focused button. Explain how the button could become disabled while its label still shows the starting count. Restore the missing update in the correct position, then run the complete repair through the same two input actions and extra disabled-click attempt. Close the window and explain the actual symptom and repair.

This sample is for repair:

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```


My predicted faulty label after each action:

Why the button and label disagree:

My restored statement and position:

Actual repaired pointer, Space and disabled-click observations:

Native close and post-run explanation:

<details>
<summary>Show answer</summary>

The faulty draft initially prints `Counter ready: 2` and shows `Remaining: 2`. After the first click, the label still reads `Remaining: 2`; after the focused Space activation, the button disables but the label still reads `Remaining: 2`. The model method and disable condition remain present, while the Label never receives new text. Restore `status.setText("Remaining: " + model.getRemaining());` after `model.reserve()` and before `reserve.setDisable(...)`. The complete repair is the prediction program: the Label progresses to `Remaining: 1` and then `Remaining: 0`, and the extra click on the disabled control changes nothing. The visible stale text is the concrete fault; normal completion of the notebook cell does not establish that later events update the display correctly.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class SeatCounter {
    private int remaining;
    public SeatCounter(int remaining) { this.remaining = remaining; }
    public int getRemaining() { return remaining; }
    public void reserve() {
        if (remaining > 0) { remaining--; }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    SeatCounter model = new SeatCounter(2);
    Label status = new Label("Remaining: " + model.getRemaining());
    Button reserve = new Button("Reserve one");
    reserve.setOnAction(event -> {
        model.reserve();
        status.setText("Remaining: " + model.getRemaining());
        reserve.setDisable(model.getRemaining() == 0);
    });
    VBox root = new VBox(12, status, reserve);
    root.setPadding(new Insets(20));
    stage.setTitle("Reservation Desk");
    stage.setScene(new Scene(root, 380, 220));
    stage.show();
    System.out.println("Counter ready: " + model.getRemaining());
});
```

Expected output:

```text
Counter ready: 2
```

Common error: Changing the initial Label text instead of refreshing it in the callback. Refreshing the Label before calling the model method. Assuming a normally completed cell proves all future button behavior.

</details>


## Independent Practice

### Build a points counter with reset

Create `PointCounter` with private points initially 0, `getPoints`, `addTwo` that stops at 4, and `reset` to 0. Build a Point Counter window with a 380 by 260 Scene, a VBox gap of 12 and padding of 20. Display `Points: n` followed by Add two and Reset buttons. Each handler must change the model and then refresh the Label. Disable Add two at 4; Reset must restore 0 and re-enable Add two. Include all required imports and the complete model class, and use the supplied `Fx.closeWindows` and `Fx.run` operations. Show the window and print only the initial report `Counter ready: 0`. Test two pointer activations, another click attempt while Add two is disabled, Reset, and Space on the keyboard-focused Add two button. After Reset, Shift+Tab moves focus from Reset back to Add two. Close the window. Explain the model change separately from the Label refresh.

My model and handler plan:

My predicted initial state and action sequence:

My complete Java program is in the work cell.

Actual pointer, disabled-click, reset and keyboard results:

Model update versus Label refresh:

Native close and post-run explanation:


### Test reset before and after an addition

Keep the original predictions and observations from the required full sequence. Recreate the complete Point Counter program for each additional test. First click Reset immediately at zero, then move focus from Reset to Add two with Shift+Tab and press Space. In a second fresh window, click Add two once, click Reset, then click Add two again. Predict and record the Label text and Add two state after every action, and close each window. Explain what these tests add to the earlier test of Reset at the upper limit. Distinguish the fixed initial console line from the later UI results.

Required full sequence retained:

Fresh reset-at-zero prediction and actual result:

Fresh add-reset-add prediction and actual result:

What each boundary test checks:

Initial stdout versus later UI states:

Native close for each test and post-run explanation:


<details>
<summary>Show answer</summary>

The complete program initially prints `Counter ready: 0` and shows `Points: 0`; both buttons are enabled. The first two Add two clicks update the model and Label to 2 and 4. At 4, Add two is disabled, and another native click does not change the display. Reset changes the model to 0, refreshes the Label to `Points: 0` and re-enables Add two. With Reset focused after its click, Shift+Tab selects Add two; Space reaches the same add handler and displays `Points: 2`. No handler prints another initial report. The private model holds points, while each control update explicitly refreshes what the learner sees. Native close removes the window. Testing Reset at zero checks that it leaves a valid initial state usable. Testing Reset after one addition checks that reset works before the upper limit, too. Recreating the whole program gives each test its own model, controls and window.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
class PointCounter {
    private int points;
    public PointCounter() { points = 0; }
    public int getPoints() { return points; }
    public void addTwo() {
        if (points < 4) { points += 2; }
    }
    public void reset() { points = 0; }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    PointCounter model = new PointCounter();
    Label status = new Label("Points: " + model.getPoints());
    Button add = new Button("Add two");
    Button reset = new Button("Reset");
    add.setOnAction(event -> {
        model.addTwo();
        status.setText("Points: " + model.getPoints());
        add.setDisable(model.getPoints() == 4);
    });
    reset.setOnAction(event -> {
        model.reset();
        status.setText("Points: " + model.getPoints());
        add.setDisable(false);
    });
    VBox root = new VBox(12, status, add, reset);
    root.setPadding(new Insets(20));
    stage.setTitle("Point Counter");
    stage.setScene(new Scene(root, 380, 260));
    stage.show();
    System.out.println("Counter ready: " + model.getPoints());
});
```

Expected output:

```text
Counter ready: 0
```

Common error: Resetting the model without refreshing the Label. Resetting to zero but leaving Add two disabled. Updating the Label without updating the model.

</details>


## Summary

A Button action can come from pointer or keyboard activation. Register its handler to run later. Keep application state in a model, then refresh controls from that state after each action. Enable or disable controls according to the available operation, and keep model guards that protect valid state.

Close the answers. Explain registration versus execution, model versus label, and focus versus enablement. Trace all three steps of the reserve handler.


## Reflection

Describe a limited campus resource with an Add or Use action and a Reset or Return action. State its model fields, boundary rule, visible feedback, and enablement decisions. Explain one test you would run with the keyboard and one direct model test.

**My design and explanation:**

Next, you will read editable field text, reject invalid values without losing valid model state, and show useful correction messages.


## Supplemental Reading

- [JavaFX 21 ButtonBase API](https://openjfx.io/javadoc/21/javafx.controls/javafx/scene/control/ButtonBase.html) documents action-handler registration.
- [JavaFX 21 Button API](https://openjfx.io/javadoc/21/javafx.controls/javafx/scene/control/Button.html) describes button activation.
- [JavaFX 21 EventHandler API](https://openjfx.io/javadoc/21/javafx.base/javafx/event/EventHandler.html) defines the event callback contract.
- [JavaFX 21 Node API](https://openjfx.io/javadoc/21/javafx.graphics/javafx/scene/Node.html) documents disable and focus behavior.
